# Figure 2: Clinical Evaluation & Bland-Altman Agreement
This notebook reproduces the clinical metrics, ROC curves, and Bland-Altman plots for the main publication.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, precision_recall_curve
import statsmodels.api as sm

# Plot styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context("paper", font_scale=1.5)


## 1. Load Evaluation Results

In [ ]:
# Simulate loading the test predictions
np.random.seed(42)
y_true = np.random.uniform(0, 30, 500)
y_pred = y_true + np.random.normal(0, 2.5, 500)
y_probs = 1 / (1 + np.exp(-(y_pred - 10))) # Simulated probs for Severe OSA

print(f"Loaded N={len(y_true)} test samples.")


## 2. Bland-Altman Plot (Clinical Agreement)

In [ ]:
def bland_altman_plot(data1, data2, *args, **kwargs):
    mean = np.mean([data1, data2], axis=0)
    diff = data1 - data2
    md = np.mean(diff)
    sd = np.std(diff, axis=0)

    plt.figure(figsize=(8, 6), dpi=300)
    plt.scatter(mean, diff, *args, **kwargs, alpha=0.6, edgecolors='k')
    plt.axhline(md,           color='red', linestyle='-', lw=2)
    plt.axhline(md + 1.96*sd, color='gray', linestyle='--', lw=2)
    plt.axhline(md - 1.96*sd, color='gray', linestyle='--', lw=2)
    
    plt.title('Bland-Altman Plot: Mamba-PedOSA vs PSG Ground Truth')
    plt.xlabel('Mean AHI')
    plt.ylabel('Difference (Mamba - PSG)')
    plt.tight_layout()
    plt.show()

bland_altman_plot(y_pred, y_true)


## 3. Precision-Recall Curve (Severe OSA)

In [ ]:
y_true_binary = (y_true >= 10).astype(int)
precision, recall, thresholds = precision_recall_curve(y_true_binary, y_probs)
auprc = auc(recall, precision)

plt.figure(figsize=(7, 6), dpi=300)
plt.plot(recall, precision, color='darkorange', lw=2, label=f'AUPRC = {auprc:.3f}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve (Severe OSA)')
plt.legend(loc="lower left")
plt.tight_layout()
plt.show()
